In [ ]:
import sys
import warnings
import torch
import tonic.datasets
import torch.nn             as nn
import torch.nn.functional  as F
import snntorch             as snn
import matplotlib.pyplot    as plt
import numpy                as np
import tonic.datasets as tonic_datasets

from tonic                  import transforms as tonic_transforms
from torch.utils.data       import DataLoader
from tqdm                   import tqdm
from pathlib                import Path

# =============================================================
# General setup
# =============================================================
USE_SC_DECAY = True
SC_LENGTH    = 256
warnings.filterwarnings("ignore")
plt.style.use('dark_background')

# =============================================================
# Configuration
# =============================================================
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

LENGTH      = 32
INPUT_SIZE  = 256
HIDDEN_SIZE = 256

# DVS-Lip has 100 classes
OUTPUT_SIZE = 100

TIME_STEPS  = 10
BATCH_SIZE  = 128
EPOCHS      = 1
LR          = 1e-3

THRESHOLD   = torch.tensor(1.0 , device=DEVICE)
BETA        = torch.tensor(0.98, device=DEVICE)
ALPHA       = torch.tensor(0.90, device=DEVICE)

# =============================================================
# Fixed-point state config (used in VerilogNeuron)
# =============================================================
USE_FIXED_STATES = True
FIXED_TOTAL_BITS = 16
FIXED_FRAC_BITS  = 7
FIXED_ADD_MODE   = "saturate"   # "saturate" or "wrap"

def events_to_frames(events, sensor_hw=(128, 128), time_window_us=50_000):
    """
    Convert events -> frames [T, 2, H, W] (polarity channels).
    Works with common spikedata event formats:
      - numpy array (N,4): [t, x, y, p]
      - structured array with fields t/x/y/p
      - dict with keys t/x/y/p (or ts/xs/ys/ps)
    """
    H, W = sensor_hw

    # --- unpack events ---
    if isinstance(events, dict):
        t = np.asarray(events.get("t", events.get("ts")))
        x = np.asarray(events.get("x", events.get("xs")))
        y = np.asarray(events.get("y", events.get("ys")))
        p = np.asarray(events.get("p", events.get("ps", events.get("polarity"))))
    else:
        ev = np.asarray(events)
        if ev.dtype.names is not None:
            t = np.asarray(ev["t"] if "t" in ev.dtype.names else ev["ts"])
            x = np.asarray(ev["x"])
            y = np.asarray(ev["y"])
            p = np.asarray(ev["p"] if "p" in ev.dtype.names else ev["polarity"])
        else:
            if ev.ndim != 2 or ev.shape[1] < 4:
                raise ValueError(f"Unexpected events shape: {ev.shape} (expected Nx4: t,x,y,p)")
            t, x, y, p = ev[:, 0], ev[:, 1], ev[:, 2], ev[:, 3]

    t = t.astype(np.int64)
    x = x.astype(np.int64)
    y = y.astype(np.int64)
    p = p.astype(np.int64)

    # polarity to {0,1}
    if p.min() < 0:
        p = (p > 0).astype(np.int64)
    else:
        p = np.clip(p, 0, 1)

    # clamp coords
    x = np.clip(x, 0, W - 1)
    y = np.clip(y, 0, H - 1)

    # time binning
    if t.size == 0:
        return np.zeros((1, 2, H, W), dtype=np.float32)

    t0 = int(t.min())
    bins = ((t - t0) // int(time_window_us)).astype(np.int64)
    T = int(bins.max() + 1)

    frames = np.zeros((T, 2, H, W), dtype=np.float32)
    np.add.at(frames, (bins, p, y, x), 1.0)
    return frames


# =============================================================
# -------- Stochastic Computing helpers (unipolar, vectorized) --------
# =============================================================
def sc_unipolar_mul(a: torch.Tensor, b: torch.Tensor, L: int = SC_LENGTH) -> torch.Tensor:
    a = a.clamp(0.0, 1.0)
    b = b.clamp(0.0, 1.0)
    shape = torch.broadcast_shapes(a.shape, b.shape)
    a_exp = a.expand(shape)
    b_exp = b.expand(shape)
    U1 = torch.rand((L,) + shape, device=a.device)
    U2 = torch.rand((L,) + shape, device=a.device)
    S1 = (U1 < a_exp)
    S2 = (U2 < b_exp)
    AND = (S1 & S2).float()
    return AND.mean(dim=0)

def sc_unipolar_scale(x: torch.Tensor, s: torch.Tensor, L: int = SC_LENGTH) -> torch.Tensor:
    return sc_unipolar_mul(x, s, L)

def to_unipolar(x: torch.Tensor, lo: float = 0.0, hi: float = 1.0) -> torch.Tensor:
    return ((x - lo) / max(1e-8, hi - lo)).clamp(0.0, 1.0)

def from_unipolar(p: torch.Tensor, lo: float = 0.0, hi: float = 1.0) -> torch.Tensor:
    return p * (hi - lo) + lo

# =============================================================
# Rate encoder
# =============================================================
def stochastic_rate_encode(x: torch.Tensor, num_steps: int = TIME_STEPS) -> torch.Tensor:
    # x: [B, N] in [0,1]
    x = x.float().unsqueeze(1).expand(-1, num_steps, -1)  # [B, T, N]
    return (torch.rand_like(x) < x).float().permute(1, 0, 2)  # [T, B, N]

def set_seed(seed: int = 0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)

# =============================================================
# COE + fixed helpers (unchanged)
# =============================================================
def tensor_to_coe(t: torch.Tensor,
                  path: str | Path = "init.coe",
                  bit_width: int | None = 16,
                  radix: int = 2,
                  per_line: int = 16,
                  flatten: str = "row"):
    if t.is_cuda:
        t = t.cpu()
    if torch.is_floating_point(t):
        raise TypeError("Input tensor must be integer-typed.")
    t = t.to(torch.int64)

    if bit_width is None:
        mins = int(t.min().item())
        maxs = int(t.max().item())
        def fits(n):
            return (mins >= -(1 << (n - 1))) and (maxs < (1 << (n - 1)))
        for n in (8, 16, 32, 64):
            if fits(n):
                bit_width = n
                break
        else:
            raise ValueError("Values do not fit in 64-bit signed range.")

    if radix not in (2, 16):
        raise ValueError("radix must be 2 (binary) or 16 (hex).")

    mask = (1 << bit_width) - 1
    if t.ndim == 1:
        flat = t
    else:
        flat = t.reshape(-1) if flatten == "row" else t.transpose(0, 1).reshape(-1)

    if radix == 2:
        fmt_width = bit_width
        to_str = lambda v: format((int(v) & mask), f"0{fmt_width}b")
    else:
        fmt_width = (bit_width + 3) // 4
        to_str = lambda v: format((int(v) & mask), f"0{fmt_width}X")

    vals = [to_str(v) for v in flat]

    header = [
        f"memory_initialization_radix={radix};",
        "memory_initialization_vector="
    ]
    body_lines = []
    for i in range(0, len(vals), per_line):
        chunk = vals[i:i+per_line]
        is_last = (i + per_line) >= len(vals)
        if is_last:
            body_lines.append(", ".join(chunk) + ";")
        else:
            body_lines.append(", ".join(chunk) + ",")

    text = "\n".join(header + body_lines) + "\n"

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return str(path), bit_width, len(vals)

def float_to_fixed(x: torch.Tensor, total_bits: int = 16, frac_bits: int = 15) -> torch.Tensor:
    max_val = (1 << (total_bits - 1)) - 1
    min_val = -(1 << (total_bits - 1))
    scale = 1 << frac_bits
    fixed = (x * scale).round().clamp(min_val, max_val)
    return fixed.to(torch.int32)

def fixed_to_float(x: torch.Tensor, frac_bits: int = 15) -> torch.Tensor:
    return x.to(torch.float32) / float(1 << frac_bits)

def _signed_limits(nbits: int):
    if nbits < 2:
        raise ValueError("nbits must be >= 2 for signed two's complement.")
    maxv = (1 << (nbits - 1)) - 1
    minv = -(1 << (nbits - 1))
    return minv, maxv

def _wrap_to_nbits(x: torch.Tensor, nbits: int) -> torch.Tensor:
    mask = (1 << nbits) - 1
    x = x.to(torch.int64) & mask
    sign_bit = 1 << (nbits - 1)
    x = torch.where(x >= sign_bit, x - (1 << nbits), x)
    return x.to(torch.int64)

def fixed_add(a_int: torch.Tensor, b_int: torch.Tensor, nbits: int, mode: str = "saturate") -> torch.Tensor:
    a_int = a_int.to(torch.int64)
    b_int = b_int.to(torch.int64)
    s = a_int + b_int

    if mode == "wrap":
        return _wrap_to_nbits(s, nbits).to(torch.int32)

    if mode == "saturate":
        minv, maxv = _signed_limits(nbits)
        return s.clamp(minv, maxv).to(torch.int32)

    raise ValueError("mode must be 'saturate' or 'wrap'")

def fixed_sub(a_int: torch.Tensor, b_int: torch.Tensor, nbits: int, mode: str = "saturate") -> torch.Tensor:
    return fixed_add(a_int, -b_int.to(torch.int64), nbits=nbits, mode=mode)

# =============================================================
# DVSGesture -> static 256 dataset wrapper
# =============================================================
class DVSGestureAsStatic256(torch.utils.data.Dataset):
    """
    Returns:
      x: torch.float32 [256] in [0,1]
      y: int label (0..10)
    """
    def __init__(self, root="./dataset", train=True, time_window_us=50_000,
                 target_hw=(16,16), normalize="max", aggregate="mean"):
        super().__init__()
        sensor_size = tonic.datasets.DVSGesture.sensor_size
        to_frame = tonic_transforms.ToFrame(sensor_size=sensor_size, time_window=time_window_us)

        self.ds = tonic.datasets.DVSGesture(
            save_to=root,
            train=train,
            transform=to_frame
        )
        self.HW = target_hw
        self.normalize = normalize
        self.aggregate = aggregate

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        frames_np, label = self.ds[idx]   # numpy: [T,2,H,W]
        frames = torch.from_numpy(frames_np).float()

        x = frames[:, 0] + frames[:, 1]   # [T,H,W]
        x = x.unsqueeze(1)                # [T,1,H,W]
        x = F.interpolate(x, size=self.HW, mode="bilinear", align_corners=False)  # [T,1,16,16]
        x = x.squeeze(1)                  # [T,16,16]

        if self.aggregate == "mean":
            x = x.mean(dim=0)
        elif self.aggregate == "sum":
            x = x.sum(dim=0)
        else:
            raise ValueError("aggregate must be 'mean' or 'sum'")

        if self.normalize == "max":
            m = x.max().clamp(min=1.0)
            x = x / m
        elif self.normalize == "sum":
            s = x.sum().clamp(min=1.0)
            x = x / s
        else:
            raise ValueError("normalize must be 'max' or 'sum'")

        x = x.reshape(-1).clamp(0.0, 1.0)  # [256]
        return x, int(label)

# =============================================================
# Learnable neuron wrappers (software)
# =============================================================
class LearnableLIF(snn.Leaky):
    def __init__(self, beta_init: torch.Tensor):
        super().__init__(beta=beta_init)
        self.beta = nn.Parameter(beta_init.clone().detach().to(DEVICE))

    def forward(self, input_, mem):
        self.beta.data.clamp_(0.01, 0.99)
        return super().forward(input_, mem)

class LearnableSynaptic(snn.Synaptic):
    def __init__(self, alpha_init: torch.Tensor, beta_init: torch.Tensor):
        super().__init__(alpha=alpha_init, beta=beta_init)
        self.alpha = nn.Parameter(alpha_init.clone().detach().to(DEVICE))
        self.beta  = nn.Parameter(beta_init.clone().detach().to(DEVICE))

    def forward(self, input_, syn, mem):
        self.alpha.data.clamp_(0.01, 0.99)
        self.beta.data.clamp_(0.01, 0.99)
        return super().forward(input_, syn, mem)

def return_model(MODEL_NAME: str, **kwargs) -> nn.Module:
    name      = MODEL_NAME.strip().lower()
    beta      = kwargs.get("BETA", torch.tensor(0.95, device=DEVICE))
    alpha     = kwargs.get("ALPHA", torch.tensor(1.00, device=DEVICE))
    threshold = kwargs.get("THRESHOLD", torch.tensor(1.00, device=DEVICE))

    if name == 'lif':
        return LearnableLIF(beta_init=beta)
    elif name == 'synaptic':
        return LearnableSynaptic(alpha_init=alpha, beta_init=beta)
    elif name == 'if':
        return snn.Leaky(beta=torch.as_tensor(1.0, device=DEVICE), threshold=threshold, reset_mechanism="zero")
    else:
        raise ValueError(f"Unknown model: {MODEL_NAME}")

# =============================================================
# Software SNN (reference)
# =============================================================
class SoftwareSNN(nn.Module):
    def __init__(self, INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, MODEL_NAME,
                 BETA_INIT=BETA, ALPHA_INIT=ALPHA, THRESHOLD=THRESHOLD):
        super().__init__()
        self.MODEL_NAME = MODEL_NAME.strip().lower()

        self.lif1 = return_model(MODEL_NAME=self.MODEL_NAME, BETA=BETA_INIT, ALPHA=ALPHA_INIT, THRESHOLD=THRESHOLD)
        self.lif2 = return_model(MODEL_NAME=self.MODEL_NAME, BETA=BETA_INIT, ALPHA=ALPHA_INIT, THRESHOLD=THRESHOLD)

        self.fc1  = nn.Linear(INPUT_SIZE, HIDDEN_SIZE, bias=False)
        self.fc2  = nn.Linear(HIDDEN_SIZE, OUTPUT_SIZE, bias=False)

    def forward(self, x: torch.Tensor, stdp: bool = False, lr: float = 1e-3) -> torch.Tensor:
        # x: [T, B, N]
        B = x.size(1)
        if self.MODEL_NAME == 'synaptic':
            syn1 = torch.zeros(B, self.fc1.out_features, device=x.device)
            mem1 = torch.zeros(B, self.fc1.out_features, device=x.device)
            syn2 = torch.zeros(B, self.fc2.out_features, device=x.device)
            mem2 = torch.zeros(B, self.fc2.out_features, device=x.device)
        else:
            mem1 = torch.zeros(B, self.fc1.out_features, device=x.device)
            mem2 = torch.zeros(B, self.fc2.out_features, device=x.device)

        spk2_rec  = []
        spk1_prev = torch.zeros(B, self.fc1.out_features, device=x.device)
        spk2_prev = torch.zeros(B, self.fc2.out_features, device=x.device)

        for t in range(x.size(0)):
            cur1 = self.fc1(x[t])
            if self.MODEL_NAME == 'synaptic':
                spk1, syn1, mem1 = self.lif1(cur1, syn1, mem1)
            else:
                spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.fc2(spk1)
            if self.MODEL_NAME == 'synaptic':
                spk2, syn2, mem2 = self.lif2(cur2, syn2, mem2)
            else:
                spk2, mem2 = self.lif2(cur2, mem2)

            spk2_rec.append(spk2)

            if stdp:
                pre, post           = spk1.detach(), spk2.detach()
                pre_prev, post_prev = spk1_prev.detach(), spk2_prev.detach()
                dw = lr * (torch.bmm(post_prev.unsqueeze(2), pre.unsqueeze(1)) -
                           torch.bmm(post.unsqueeze(2), pre_prev.unsqueeze(1)))
                self.fc2.weight.data += dw.mean(0)
                self.fc2.weight.data.clamp_(-1.0, 1.0)
                spk1_prev = spk1
                spk2_prev = spk2

        return torch.stack(spk2_rec)  # [T, B, C]

# =============================================================
# VerilogNeuron + HardwareSNN (copied structure from your code)
# =============================================================
class VerilogNeuron(nn.Module):
    def __init__(self, threshold: float = 1.0, alpha: float = 0.9, beta: float = 0.9,
                 mode: str = "lif", reset_to_zero: bool = True,
                 use_sc: bool = USE_SC_DECAY, sc_length: int = SC_LENGTH,
                 use_fixed_states: bool = USE_FIXED_STATES,
                 nbits: int = FIXED_TOTAL_BITS, frac_bits: int = FIXED_FRAC_BITS,
                 add_mode: str = FIXED_ADD_MODE):
        super().__init__()
        self.threshold = float(threshold)
        self.alpha = float(alpha)
        self.beta = float(beta)
        self.mode = mode
        self.reset_to_zero = reset_to_zero
        self.use_sc = use_sc
        self.sc_length = sc_length
        self.use_fixed = use_fixed_states
        self.nbits = int(nbits)
        self.frac_bits = int(frac_bits)
        self.add_mode = str(add_mode)

    def _decay(self, v: torch.Tensor, alpha: float) -> torch.Tensor:
        if not self.use_sc:
            return alpha * v
        v_u = to_unipolar(v, 0.0, self.threshold)
        a_t = torch.as_tensor(alpha, device=v.device, dtype=v.dtype)
        dec_u = sc_unipolar_scale(v_u, a_t, self.sc_length)
        return from_unipolar(dec_u, 0.0, self.threshold)

    def _i_decay(self, I: torch.Tensor, beta: float) -> torch.Tensor:
        if not self.use_sc:
            return beta * I
        I_u = to_unipolar(I, 0.0, self.threshold)
        b_t = torch.as_tensor(beta, device=I.device, dtype=I.dtype)
        dec_u = sc_unipolar_scale(I_u, b_t, self.sc_length)
        return from_unipolar(dec_u, 0.0, self.threshold)

    def forward(self, x: torch.Tensor, v_mem: torch.Tensor, I_t: torch.Tensor | None = None):
        if not self.use_fixed:
            if self.mode == "if":
                v_next = v_mem + x
            elif self.mode == "lif":
                v_next = (v_mem * self.alpha) + x
            elif self.mode == "synaptic":
                I_next = (I_t * self.beta) + x
                v_next = (v_mem * self.alpha) + I_next
            else:
                raise ValueError("mode must be 'if' | 'lif' | 'synaptic'")

            v_next = v_next.clamp(0.0, self.threshold)
            spk = (v_next >= self.threshold).float()
            if self.reset_to_zero:
                v_next = torch.where(spk.bool(), torch.zeros_like(v_next), v_next)
            else:
                v_next = torch.where(spk.bool(), v_next - self.threshold, v_next)

            if self.mode == "synaptic":
                I_next = torch.where(spk.bool(), torch.zeros_like(I_next), I_next)
                return spk, v_next, I_next
            return spk, v_next

        scale = 1 << self.frac_bits
        thr_q = int(round(self.threshold * scale))

        x_q = float_to_fixed(x, total_bits=self.nbits, frac_bits=self.frac_bits)
        v_q = float_to_fixed(v_mem, total_bits=self.nbits, frac_bits=self.frac_bits)

        if self.mode == "if":
            v_next_q = fixed_add(v_q, x_q, nbits=self.nbits, mode=self.add_mode)

        elif self.mode == "lif":
            vdec_f   = self._decay(v_mem, self.alpha)
            vdec_q   = float_to_fixed(vdec_f, total_bits=self.nbits, frac_bits=self.frac_bits)
            v_next_q = fixed_add(vdec_q, x_q, nbits=self.nbits, mode=self.add_mode)

        elif self.mode == "synaptic":
            Idec_f   = self._i_decay(I_t, self.beta)
            Idec_q   = float_to_fixed(Idec_f, total_bits=self.nbits, frac_bits=self.frac_bits)
            I_next_q = fixed_add(Idec_q, x_q, nbits=self.nbits, mode=self.add_mode)

            vdec_f   = self._decay(v_mem, self.alpha)
            vdec_q   = float_to_fixed(vdec_f, total_bits=self.nbits, frac_bits=self.frac_bits)
            v_next_q = fixed_add(vdec_q, I_next_q, nbits=self.nbits, mode=self.add_mode)
        else:
            raise ValueError("mode must be 'if' | 'lif' | 'synaptic'")

        v_next_q = v_next_q.clamp(min=0, max=thr_q)
        spk = (v_next_q >= thr_q).float()

        if self.reset_to_zero:
            v_next_q = torch.where(spk.bool(), torch.zeros_like(v_next_q), v_next_q)
        else:
            v_next_q = torch.where(
                spk.bool(),
                fixed_sub(v_next_q, torch.as_tensor(thr_q, dtype=torch.int32, device=v_next_q.device),
                          nbits=self.nbits, mode=self.add_mode),
                v_next_q
            )

        v_next = fixed_to_float(v_next_q, frac_bits=self.frac_bits)

        if self.mode == "synaptic":
            I_next_q = torch.where(spk.bool(), torch.zeros_like(v_next_q), I_next_q)
            I_next = fixed_to_float(I_next_q, frac_bits=self.frac_bits)
            return spk, v_next, I_next

        return spk, v_next

class HardwareSNN(nn.Module):
    def __init__(self, fc1_weight: torch.Tensor, fc2_weight: torch.Tensor,
                 beta: torch.Tensor, alpha: torch.Tensor, threshold: torch.Tensor,
                 mode: str = 'lif', sc_length: int = SC_LENGTH, nbit: int = FIXED_TOTAL_BITS, frac_bit: int = FIXED_FRAC_BITS):
        super().__init__()
        self.register_buffer('fc1_weight_q', float_to_fixed(fc1_weight, 8, 7))
        self.register_buffer('fc2_weight_q', float_to_fixed(fc2_weight, 8, 7))

        self.alpha = float(alpha)
        self.beta = float(beta)
        self.threshold = float(threshold)
        self.mode = mode.strip().lower()

        self.hidden_neurons = nn.ModuleList([
            VerilogNeuron(threshold=self.threshold, alpha=self.alpha, beta=self.beta, mode=self.mode,
                          sc_length=sc_length, nbits=nbit, frac_bits=frac_bit)
            for _ in range(HIDDEN_SIZE)
        ])
        self.output_neurons = nn.ModuleList([
            VerilogNeuron(threshold=self.threshold, alpha=self.alpha, beta=self.beta, mode=self.mode,
                          sc_length=sc_length, nbits=nbit, frac_bits=frac_bit)
            for _ in range(OUTPUT_SIZE)
        ])

    def forward(self, x_spikes: torch.Tensor) -> torch.Tensor:
        T, B, _ = x_spikes.shape
        out_spikes = x_spikes.new_zeros(T, B, OUTPUT_SIZE)

        v_hidden = x_spikes.new_zeros(B, HIDDEN_SIZE)
        v_out    = x_spikes.new_zeros(B, OUTPUT_SIZE)
        I_hidden = x_spikes.new_zeros(B, HIDDEN_SIZE) if self.mode == "synaptic" else None
        I_out    = x_spikes.new_zeros(B, OUTPUT_SIZE) if self.mode == "synaptic" else None

        w1 = self.fc1_weight_q.float().t() / float(1 << FIXED_FRAC_BITS)  # [N, H]
        w2 = self.fc2_weight_q.float().t() / float(1 << FIXED_FRAC_BITS)  # [H, C]

        for t in range(T):
            cur1 = x_spikes[t].float() @ w1

            h_spk = x_spikes.new_zeros(B, HIDDEN_SIZE)
            for i, neuron in enumerate(self.hidden_neurons):
                if self.mode == "synaptic":
                    spk, v_hidden[:, i], I_hidden[:, i] = neuron(cur1[:, i], v_hidden[:, i], I_hidden[:, i])
                else:
                    spk, v_hidden[:, i] = neuron(cur1[:, i], v_hidden[:, i])
                h_spk[:, i] = spk

            cur2 = h_spk.float() @ w2

            for j, neuron in enumerate(self.output_neurons):
                if self.mode == "synaptic":
                    spk, v_out[:, j], I_out[:, j] = neuron(cur2[:, j], v_out[:, j], I_out[:, j])
                else:
                    spk, v_out[:, j] = neuron(cur2[:, j], v_out[:, j])
                out_spikes[t, :, j] = spk

        return out_spikes

# =============================================================
# Training & evaluation
# =============================================================
def train(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer):
    model.train()
    for epoch in range(EPOCHS):
        running = 0.0
        for x, y in tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            x, y = x.to(DEVICE), y.to(DEVICE)      # x: [B,256] in [0,1]
            x_encoded = stochastic_rate_encode(x)  # [T,B,256]

            optimizer.zero_grad()
            logits_over_time = model(x_encoded)    # [T,B,C] spikes
            output = logits_over_time.sum(0)       # [B,C]
            loss = nn.CrossEntropyLoss()(output, y)
            loss.backward()
            optimizer.step()

            running += loss.item()
        print(f"Epoch {epoch+1}, Loss: {running/len(loader):.4f}")

def evaluate(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating"):
            x, y = x.to(DEVICE), y.to(DEVICE)
            x_encoded = stochastic_rate_encode(x)
            logits_over_time = model(x_encoded)
            output = logits_over_time.sum(0)
            pred = output.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

def eval_hw_repeated(hardware_model_fn, loader, repeats=1, base_seed=123):
    accs = []
    for r in range(repeats):
        set_seed(base_seed + r)
        accs.append(evaluate(hardware_model_fn(), loader))
    return float(np.mean(accs)), float(np.std(accs))

# =============================================================
# Data loading: DVSGesture -> [B,256] in [0,1]
# =============================================================
train_ds = DVSGestureAsStatic256(root="./dataset/DVS", train=True,
                                 time_window_us=50_000,
                                 target_hw=(16,16),
                                 normalize="max",
                                 aggregate="mean")

test_ds  = DVSGestureAsStatic256(root="./dataset/DVS", train=False,
                                 time_window_us=50_000,
                                 target_hw=(16,16),
                                 normalize="max",
                                 aggregate="mean")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)

# --- Sanity checks ---
xb, yb = next(iter(train_loader))
print("SANITY x:", xb.shape, xb.dtype, xb.min().item(), xb.max().item())  # [B,256], [0,1]
print("SANITY y:", yb.shape, int(yb.min()), int(yb.max()))                # 0..10

# =============================================================
# Run experiment
# =============================================================
MODES   = ["if", "lif", "synaptic"]
SC_LIST = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]

results = {}
REPEATS = 1

NBITS = 16
FRAC  = 7

for MODE_NAME in MODES:
    print(f"\n====================")
    print(f"MODE: {MODE_NAME.upper()}")
    print(f"====================")

    # Train software model
    software_model = SoftwareSNN(
        INPUT_SIZE=INPUT_SIZE,
        HIDDEN_SIZE=HIDDEN_SIZE,
        OUTPUT_SIZE=OUTPUT_SIZE,
        MODEL_NAME=MODE_NAME,
        BETA_INIT=BETA,
        ALPHA_INIT=ALPHA,
        THRESHOLD=THRESHOLD
    ).to(DEVICE)

    optimizer = torch.optim.Adam(software_model.parameters(), lr=LR)

    print("Training software model...")
    train(software_model, train_loader, optimizer)

    print("Evaluating software model...")
    set_seed(0)
    sw_acc = evaluate(software_model, test_loader)
    print(f"Software Accuracy ({MODE_NAME}): {sw_acc*100:.2f}%")

    L_list, acc_mean_list, acc_std_list = [], [], []

    for sc_length in SC_LIST:
        print(f"\nSC Length: {sc_length}")

        def build_hw():
            return HardwareSNN(
                fc1_weight=software_model.fc1.weight.data.detach().to(DEVICE),
                fc2_weight=software_model.fc2.weight.data.detach().to(DEVICE),
                beta=BETA, alpha=ALPHA, threshold=THRESHOLD,
                mode=MODE_NAME,
                sc_length=sc_length,
                nbit=NBITS,
                frac_bit=FRAC
            ).to(DEVICE)

        mean_acc, std_acc = eval_hw_repeated(build_hw, test_loader, repeats=REPEATS, base_seed=1000+sc_length)
        print(f"Hardware Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")

        L_list.append(sc_length)
        acc_mean_list.append(mean_acc)
        acc_std_list.append(std_acc)

    results[MODE_NAME] = {
        "software_acc": sw_acc,
        "L": L_list,
        "mean": acc_mean_list,
        "std": acc_std_list
    }

print("\nDone. Results keys:", results.keys())

# =============================================================
# Display software / hardware accuracy
# =============================================================
plt.figure(figsize=(8,5))
for mode, c in zip(MODES, ["#0d00ffff", "#ff4d00ff"]):
    L = np.array(results[mode]["L"])
    mean = np.array(results[mode]["mean"])
    std = np.array(results[mode]["std"])

    plt.plot(L, mean*100, marker="o", label=f"{mode.upper()} HW", color=c)
    plt.fill_between(L, (mean-std)*100, (mean+std)*100, alpha=0.2)

for mode, c in zip(MODES, ["#0d00ffff", "#ff4d00ff"]):
    plt.axhline(results[mode]["software_acc"]*100, linestyle="--", alpha=0.6, label=f"{mode.upper()} SW", color=c)

plt.xscale("log", base=2)
plt.xlabel("SC bitstream length (L)")
plt.ylabel("Accuracy (%)")
plt.title("Hardware Accuracy vs. Stochastic Stream Length (per neuron mode)")
plt.grid(True, which="both", alpha=0.3)
plt.legend()

plt.savefig(
    "SW_acc_HW_acc_DVS.svg",
    format="svg",
    transparent=True,
    bbox_inches="tight",
    pad_inches=0
)

plt.show()

plt.figure(figsize=(8,5))
for mode, c in zip(MODES, ["#0d00ffff", "#ff4d00ff"]):
    L = np.array(results[mode]["L"])
    drop = (results[mode]["software_acc"] - np.array(results[mode]["mean"])) * 100
    plt.plot(L, drop, marker="o", label=f"{mode.upper()} (SW-HW)", color=c)

plt.xscale("log", base=2)
plt.xlabel("SC bitstream length (L)")
plt.ylabel("Accuracy drop (percentage points)")
plt.title("Accuracy loss due to SC approximation")
plt.grid(True, which="both", alpha=0.3)
plt.legend()

plt.savefig(
    "SW_acc_HW_acc_loss_DVS.svg",
    format="svg",
    transparent=True,
    bbox_inches="tight",
    pad_inches=0
)

plt.show()